# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and includes multiple record sets and fields describing clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset-level metadata
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Date published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Keywords: {getattr(metadata, 'keywords', 'N/A')}")

## 2. Data Overview
Review the available record sets, their `@id`s, and fields as described by the Croissant schema.

In [ ]:
# List all available record sets and their @id
# The Croissant 'dataset' object exposes record sets via `dataset.record_sets` (mlcroissant >= 0.7.0)

record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")

for rs in record_sets:
    print(f"RecordSet Name: {rs.name}\nRecordSet @id: {rs.id}\nDescription: {getattr(rs, 'description', 'No description')}\nFields:")
    for field in rs.fields:
        print(f"  - {field.name} (field @id: {field.id}) | Type: {field.data_type}")
    print("\n---\n")

## 3. Data Extraction
We'll load data from the main record set(s) into pandas DataFrames for further analysis. Here, all entities are referenced by their `@id`. Please adjust the variable `main_record_set_id` if a different record set is preferred.

In [ ]:
# Prepare to extract data from each record set, referenced by @id
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    dataframes[rs_id] = df
    print(f"DataFrame shape: {df.shape}\n")

# If there is only one record set, we will use that; otherwise, select the most relevant one
if len(record_set_ids) == 1:
    main_record_set_id = record_set_ids[0]
else:
    # Optionally, choose the main table by name or by inspecting available record sets
    main_record_set_id = record_set_ids[0]  # Replace with the appropriate @id if needed

main_df = dataframes[main_record_set_id]

print(f"Fields (columns) in main record set (@id: {main_record_set_id}):\n{main_df.columns.tolist()}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps. We'll filter records based on a numeric field (e.g., Age), normalize values, and group by another key attribute (e.g., Sex or Tumor Location). All columns will be referenced by their `@id` as defined in the record set fields above.

In [ ]:
# Pick numeric field and group field for analysis.
# Replace these @id values with those from the field list above if different in your dataset.
# (For this dataset, suppose age is stored at field @id 'age' and sex as 'sex' -- double check your record set overview!)

# Let's try to autodetect a likely age/sex field for demonstration:

import re
age_col_candidates = [col for col in main_df.columns if re.search('age', col, re.IGNORECASE)]
sex_col_candidates = [col for col in main_df.columns if re.search('sex|gender', col, re.IGNORECASE)]

# Fallbacks if not found
numeric_field_id = age_col_candidates[0] if age_col_candidates else main_df.columns[0]  # replace by appropriate @id
group_field_id = sex_col_candidates[0] if sex_col_candidates else main_df.columns[1]   # replace by appropriate @id

print(f"Using numeric field '@id': {numeric_field_id}")
print(f"Using group field '@id': {group_field_id}")

# Filter records with age > 60 (if appropriate) as an example threshold
try:
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    threshold = 60
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} of {len(main_df)}")
    display(filtered_df.head())

    # Normalize the age field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the selected group field and compute the mean of numerical columns
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id, dropna=False).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id} (means):")
        display(grouped_df.head())
except Exception as e:
    print(f"EDA step encountered an issue: {e}")

## 5. Visualization
Visualize the distribution of the selected numeric field, and the breakdown by the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

if group_field_id in main_df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² dataset using `mlcroissant`. Starting from the Croissant schema, we loaded the metadata, explored record sets, extracted data by referencing all entities via their `@id`, and performed initial data analysis and visualization on demographic variables such as age and sex. This workflow demonstrates how `mlcroissant` enables transparent and reproducible access to complex FAIR datasets defined by Croissant schemas.

You can now extend this notebook with further data cleaning, analysis, predictive modeling, or exporting processed data.